# 06 — Attach Inactive Addresses

**Run this AFTER `06_Create_Addresses.ipynb` (the active one) has finished.**

Inactive subscriptions don't get a new address — each one is attached to
whichever address is *currently* the target account's default service
address (`defaultShipping = true`), via `find_default_shipping_address_id()`.

This only works correctly once every active subscription's address has
already been created, because each newly-created address supersedes the
previous default (see `add_address_to_account` in `onebill_common.py`) — the
"final" default for an account isn't settled until its last active address
exists. Running this before or during the active pass could pick up a
default that's about to be replaced.

Accounts with no active subscriptions at all (nothing to have established a
default) will fail here — there's nothing to attach to.


## 1. Setup


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("attach_inactive_addresses")

df_subscriptions = try_load_df("subscriptions_inactive_resolved", dtype=SUBSCRIPTIONS_RESOLVED_DTYPES)
if df_subscriptions is None or df_subscriptions.empty:
    logger.warning("No 'subscriptions_inactive_resolved' file found (or it's empty) — run 05_Fetch_Inactive_Subscriptions.ipynb first, or there simply were no inactive subscriptions.")
    df_subscriptions = pd.DataFrame()
else:
    logger.info(f"Loaded {len(df_subscriptions):,} inactive subscriptions")


## 2. Per-subscription default-address lookup


In [ ]:
def attach_to_default_address(session: requests.Session, row: dict) -> dict:
    subscription_id = row["SubscriptionUSN"]
    account_number  = row["TargetAccountNumber"]

    result = {
        "SubscriptionUSN":     subscription_id,
        "TargetAccountNumber": account_number,
        "status":              "failed",
        "ship_add_id":         None,
        "error":               None,
    }

    status, address_id, error = find_default_shipping_address_id(session, account_number)
    if status == "found":
        result["status"] = "reused_default"
        result["ship_add_id"] = address_id
        logger.info(f"[OK] subscription {subscription_id} (inactive) -> {account_number} reusing default address id={address_id}")
    elif status == "not_found":
        result["error"] = "account has no address with defaultShipping=true to reuse — has its active pass run yet?"
        logger.error(f"[FAIL] subscription {subscription_id} (inactive) — {result['error']}")
    else:
        result["error"] = error
        logger.error(f"[FAIL] subscription {subscription_id} (inactive) — {error}")

    return result


## 3. Run (parallel driver)


In [ ]:
def attach_all_inactive(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    session = new_session(max_workers=max_workers)
    rows = df.to_dict("records")
    total = len(rows)
    results = []
    logger.info(f"Attaching {total:,} inactive subscriptions to their account's default address, {max_workers} workers...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(attach_to_default_address, session, row): row["SubscriptionUSN"] for row in rows}
        for i, future in enumerate(as_completed(futures), start=1):
            results.append(future.result())
            if i % 50 == 0 or i == total:
                ok = sum(1 for r in results if r["status"] == "reused_default")
                logger.info(f"Progress: {i}/{total} — {ok} attached so far")
    return pd.DataFrame(results)


if df_subscriptions.empty:
    df_address_results_inactive = pd.DataFrame(columns=["SubscriptionUSN", "TargetAccountNumber", "status", "ship_add_id", "error"])
else:
    df_address_results_inactive = attach_all_inactive(df_subscriptions)

df_address_results_inactive.head(20)


## 4. Failures


In [ ]:
not_attached = df_address_results_inactive[df_address_results_inactive["status"] != "reused_default"]
print(f"{len(not_attached):,} / {len(df_address_results_inactive):,} inactive subscriptions not attached")
not_attached


## 5. Save


In [ ]:
save_df("address_results_inactive", df_address_results_inactive)
